# 质量因子研究

**因子**：ROE + ROE稳定性 + 毛利率 → 综合质量因子  
**关键处理**：shift(1) + ffill 防止前视偏差  
**分析链**：IC分析 → 分层回测 → 前视偏差讨论 → 结论

## Section 0: 参数设置

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['font.family'] = ['Arial Unicode MS', 'SimHei', 'DejaVu Sans']
matplotlib.rcParams['axes.unicode_minus'] = False

# ── 研究参数 ──────────────────────────────────────────────────────────────
START_DATE   = "2020-01-01"   # 回测起始日
END_DATE     = "2023-12-31"   # 回测结束日
PERIODS      = 12             # 财报期数（12 期 ≈ 3 年季报）
ROE_WINDOW   = 8              # 稳定性滚动窗口（期）
FORWARD_DAYS = 20             # 前向收益天数
N_GROUPS     = 10             # 分层回测组数

# 股票池（mock 数据演示用）
MOCK_SYMBOLS = [f"{i:06d}" for i in range(1, 51)]  # 50 支模拟股票

print("✅ 参数设置完成")
print(f"   研究区间: {START_DATE} ~ {END_DATE}")
print(f"   财报期数: {PERIODS} 期 | 稳定性窗口: {ROE_WINDOW} 期")
print(f"   前向收益: {FORWARD_DAYS} 日 | 分层数: {N_GROUPS}")

## Section 1: 构建 ROE 宽表（mock 数据演示）

> **注意**：本节使用 mock 财务数据演示数据结构与前视偏差处理逻辑。  
> 生产环境请替换为 `fundamental_loader.get_financials` 真实数据。

In [ ]:
# ── 生成 mock 财务数据 ────────────────────────────────────────────────────
# 需要真实数据：
#   from utils.fundamental_loader import get_financials
#   financials_dict = {sym: get_financials(sym, periods=PERIODS) for sym in symbols}

np.random.seed(42)
report_dates = pd.date_range("2017-03-31", periods=PERIODS, freq="QE")

mock_financials = {}
for sym in MOCK_SYMBOLS:
    # 每支股票随机生成 ROE（有自相关模拟真实季报）
    roe_base = np.random.uniform(0.03, 0.20)
    roe = np.clip(
        np.cumsum(np.random.normal(0, 0.02, PERIODS)) + roe_base,
        0.01, 0.35
    )
    net_margin = np.clip(
        np.cumsum(np.random.normal(0, 0.015, PERIODS)) + np.random.uniform(0.05, 0.25),
        0.01, 0.50
    )
    mock_financials[sym] = pd.DataFrame(
        {"roe": roe, "net_margin": net_margin},
        index=report_dates
    )

print(f"✅ 生成 mock 财务数据: {len(mock_financials)} 支股票 × {PERIODS} 期")
print("示例（000001）:")
mock_financials["000001"].head(4)

In [ ]:
from research.factors.quality.quality_factor import (
    compute_roe_factor,
    compute_roe_stability,
    compute_gross_margin,
    compute_composite_quality,
)

# 构建 ROE 宽表（已包含 shift(1) + ffill）
roe_wide = compute_roe_factor(mock_financials)

print(f"ROE 宽表形状: {roe_wide.shape}  （行=日期, 列=股票）")
print(f"日期范围: {roe_wide.index[0].date()} ~ {roe_wide.index[-1].date()}")
print(f"缺失值比例: {roe_wide.isnull().mean().mean():.1%}")
roe_wide.iloc[:5, :5]

In [ ]:
# 验证 shift(1) 防前视效果：对比原始数据与 roe_wide 的首个非 NaN 日期
first_report = report_dates[0]
first_valid = roe_wide["000001"].first_valid_index()

print(f"首个报告期末: {first_report.date()}")
print(f"roe_wide 首个有效值日期: {first_valid.date() if first_valid else 'None'}")
print("")
print("→ 首个报告期末当日 roe_wide 应为 NaN（因 shift(1) 丢弃了第一期）")
print(f"  验证: roe_wide.loc[first_report, '000001'] = {roe_wide.loc[first_report, '000001'] if first_report in roe_wide.index else 'NaN（日期不在索引中）'}")

## Section 2: 计算 ROE 稳定性 / 毛利率 / 合成质量因子

In [ ]:
# ── 各子因子 ──────────────────────────────────────────────────────────────
gm_wide   = compute_gross_margin(mock_financials)
stab_wide = compute_roe_stability(roe_wide, window=ROE_WINDOW)
quality   = compute_composite_quality(roe_wide, stab_wide, gm_wide)

print(f"✅ ROE 宽表:      {roe_wide.shape}")
print(f"✅ 毛利率宽表:    {gm_wide.shape}")
print(f"✅ 稳定性宽表:    {stab_wide.shape}")
print(f"✅ 综合质量因子:  {quality.shape}")

In [ ]:
# 截面分布：取某一日快照
snapshot_date = quality.index[quality.notna().any(axis=1)][100]  # 第100个有效日

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
fig.suptitle(f"各子因子截面分布（{snapshot_date.date()}）", fontsize=14)

data_list = [
    (roe_wide.loc[snapshot_date].dropna(),   "ROE",       axes[0, 0]),
    (gm_wide.loc[snapshot_date].dropna(),    "毛利率",     axes[0, 1]),
    (stab_wide.loc[snapshot_date].dropna(),  "ROE稳定性", axes[1, 0]),
    (quality.loc[snapshot_date].dropna(),    "综合质量因子", axes[1, 1]),
]

for data, title, ax in data_list:
    ax.hist(data, bins=20, edgecolor='white', alpha=0.85)
    ax.set_title(title)
    ax.set_xlabel("因子值")
    ax.set_ylabel("频次")
    ax.axvline(data.mean(), color='red', linestyle='--', label=f'均值={data.mean():.3f}')
    ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

## Section 3: IC / ICIR 分析

IC (Information Coefficient) 衡量因子对未来收益的预测能力。  
ICIR = IC均值 / IC标准差，衡量预测稳定性（>0.5 为良好，>1.0 为优秀）。

In [ ]:
# ── 构造 mock 收益数据 ────────────────────────────────────────────────────
# 需要真实数据：
#   from utils.data_loader import build_price_matrix, build_return_matrix
#   price_wide = build_price_matrix(MOCK_SYMBOLS, START_DATE, END_DATE)
#   ret_wide = build_return_matrix(price_wide)
#   ret_fwd = ret_wide.shift(-FORWARD_DAYS).rolling(FORWARD_DAYS).sum()

# mock：基于质量因子加一些噪音生成收益（仅演示流程）
trade_dates = quality.dropna(how='all').index

np.random.seed(0)
# 前向收益 = 质量因子 × 0.02 + 噪音
ret_fwd = (
    quality.loc[trade_dates]
    .mul(0.02)
    .add(pd.DataFrame(
        np.random.normal(0, 0.05, (len(trade_dates), len(MOCK_SYMBOLS))),
        index=trade_dates,
        columns=MOCK_SYMBOLS
    ))
)

print(f"前向收益矩阵形状: {ret_fwd.shape}")

In [ ]:
from utils.factor_analysis import compute_ic_series, ic_summary

# 对齐因子与收益的日期
common_idx = quality.index.intersection(ret_fwd.index)
factor_aligned = quality.loc[common_idx]
ret_aligned    = ret_fwd.loc[common_idx]

# 计算 IC 序列
ic_series = compute_ic_series(
    factor_wide=factor_aligned,
    ret_wide=ret_aligned,
    method="spearman",
    min_stocks=10,
)

summary = ic_summary(ic_series, name="综合质量因子")
print("IC 统计摘要:")
for k, v in summary.items():
    print(f"  {k:15s}: {v:.4f}" if isinstance(v, float) else f"  {k:15s}: {v}")

In [ ]:
# IC 序列可视化
fig, axes = plt.subplots(2, 1, figsize=(14, 8))

# IC 时间序列
ax = axes[0]
ic_series.plot(ax=ax, color='steelblue', alpha=0.7, linewidth=0.8, label='日IC')
ic_series.rolling(60).mean().plot(ax=ax, color='red', linewidth=1.5, label='60日滚动均值')
ax.axhline(0, color='black', linestyle='-', linewidth=0.5)
ax.axhline(ic_series.mean(), color='green', linestyle='--', linewidth=1,
           label=f'IC均值={ic_series.mean():.4f}')
ax.set_title('综合质量因子 IC 序列')
ax.set_ylabel('IC')
ax.legend()

# IC 分布直方图
ax = axes[1]
ic_series.dropna().hist(ax=ax, bins=40, edgecolor='white', alpha=0.85, color='steelblue')
ax.axvline(0, color='black', linestyle='-', linewidth=1)
ax.axvline(ic_series.mean(), color='red', linestyle='--', linewidth=1.5,
           label=f'均值={ic_series.mean():.4f}')
ax.set_title('IC 分布')
ax.set_xlabel('IC')
ax.legend()

plt.tight_layout()
plt.show()

print(f"\nIC均值: {summary['IC_mean']:.4f} | ICIR: {summary['ICIR']:.4f} | IC>0占比: {summary['pct_pos']:.1%}")

## Section 4: 分层回测（n_groups=10）

将股票按质量因子分为 10 组，观察各组累计收益是否单调递增（选股有效性验证）。

In [ ]:
from utils.factor_analysis import quintile_backtest

# 分层回测（quality 越高预期收益越高 → Qn_minus_Q1）
group_ret, ls_ret = quintile_backtest(
    factor_wide=factor_aligned,
    ret_wide=ret_aligned,
    n_groups=N_GROUPS,
    long_short="Qn_minus_Q1",
)

print(f"分层收益矩阵形状: {group_ret.shape}")
print(f"多空收益序列长度: {len(ls_ret)}")
group_ret.head(3)

In [ ]:
# 累计收益可视化
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# 各组累计收益
ax = axes[0]
cum_ret = (1 + group_ret.fillna(0)).cumprod()
for col in cum_ret.columns:
    # Q1 最低质量（红色），Qn 最高质量（绿色）
    alpha = 0.4 + 0.6 * (int(col[1:]) - 1) / (N_GROUPS - 1)
    color = 'green' if col == group_ret.columns[-1] else (
            'red'   if col == group_ret.columns[0]  else 'grey')
    cum_ret[col].plot(ax=ax, label=col, alpha=alpha, color=color,
                      linewidth=1.5 if col in [group_ret.columns[0], group_ret.columns[-1]] else 0.8)
ax.set_title(f'各分层累计收益（{N_GROUPS}组）')
ax.set_ylabel('累计收益（倍数）')
ax.legend(fontsize=7, ncol=2)
ax.axhline(1, color='black', linewidth=0.5)

# 多空组合累计收益
ax = axes[1]
cum_ls = (1 + ls_ret.fillna(0)).cumprod()
cum_ls.plot(ax=ax, color='navy', linewidth=1.5)
ax.set_title('多空组合累计收益（高质量 − 低质量）')
ax.set_ylabel('累计收益（倍数）')
ax.axhline(1, color='red', linestyle='--', linewidth=0.8)

plt.tight_layout()
plt.show()

# 各组年化收益汇总
ann_ret = (cum_ret.iloc[-1] ** (252 / len(cum_ret)) - 1)
print("各分层年化收益:")
print(ann_ret.to_frame("年化收益").T.to_string())

## Section 5: 前视偏差分析

### 财报发布日 vs 报告期末

| 报告期 | 报告期末 | 实际公告截止日 |
|--------|----------|----------------|
| Q1     | 3月31日  | 4月30日        |
| H1     | 6月30日  | 8月31日        |
| Q3     | 9月30日  | 10月31日       |
| 年报   | 12月31日 | 次年4月30日    |

> **关键认知**：如果直接用报告期末作为数据可用日，会引入 1~4 个月的前视偏差。
> 例如：年报数据的报告期是 12月31日，但实际要到次年 4 月底才能看到全部年报。
> 使用 12月31日 这个时间点的年报数据就等于「看到了未来」。

In [ ]:
print("\n前视偏差处理方法对比：\n")

print("❌ 错误做法（引入前视偏差）：")
print("   直接将报告期末作为数据可用时间点")
print("   → 2020-12-31 的年报数据在 2021-01-01 就被用于交易")
print("   → 实际年报 2021-04-30 才公布，早用了约 4 个月")
print()

print("✅ 本模块的做法（shift(1) + ffill）：")
print("   对财报序列先 shift(1)：只用上一期已公布数据")
print("   再 ffill 对齐到日频：两季报之间的交易日沿用最新已知值")
print("   → 等效于：看到 Q1 报告时，用的仍是上一期（上一季度）的财务指标")
print()

print("⚠️  更严格的做法（生产环境建议）：")
print("   使用实际公告日（announcement date）作为时间索引")
print("   需要额外获取 akshare 公告日数据")
print("   处理方法：将 df.index 替换为公告日，再 ffill")

# 演示：shift(1) 的效果
sample_sym = MOCK_SYMBOLS[0]
original_roe = mock_financials[sample_sym]["roe"]
shifted_roe  = original_roe.shift(1)

comparison = pd.DataFrame({
    "原始ROE（有前视）": original_roe,
    "shift(1)后（无前视）": shifted_roe,
})
print(f"\n{sample_sym} 前视偏差处理效果（前5期）：")
print(comparison.head(5).to_string())

## Section 6: 结论

### 因子有效性评估

| 指标 | 结果 | 评级 |
|------|------|------|
| IC 均值 | *(见上方 IC 统计)* | 需真实数据验证 |
| ICIR | *(见上方)* | 需真实数据验证 |
| IC > 0 占比 | *(见上方)* | 需真实数据验证 |
| 分层单调性 | *(见分层回测图)* | 需真实数据验证 |

### 局限性

1. **前视偏差残留**：shift(1) 仍使用报告期末为时间基准，未引入真实公告日；  
   生产环境应替换为实际公告日时间戳。

2. **财报口径**：`get_financials` 返回的是财务分析指标，并非原始财务报表；  
   部分指标（如毛利率）可能为 `net_margin` 替代，精度有限。

3. **幸存者偏差**：现有 mock 股票池未考虑退市/停牌，真实回测需用历史成分股。

4. **行业/市值中性化**：综合质量因子未做行业中性化，
   建议生产环境使用 `neutralize_factor` 处理后再评估。

5. **财报质量**：A 股财务数据存在盈余管理风险，ROE 可能受非经常性损益影响，
   建议结合经营性现金流做质量校验。

### 后续工作

- [ ] 接入真实财报数据（`fundamental_loader.get_financials`）
- [ ] 引入实际公告日防止残留前视偏差
- [ ] 行业中性化后重新评估 IC/ICIR
- [ ] 与动量因子合成多因子模型